# Byte Pair Encoding (BPE)

**Audience:** complete beginners. You do not need prior NLP.

BPE is the algorithm behind GPT-style tokenizers (and many Llama tokenizers). It starts from
tiny pieces (characters, or even bytes) and **glues the most common adjacent pair** over and
over until the vocabulary is big enough.

After this notebook you will be able to:

1. Explain why word-level vocabularies break on new words.
2. Train a tiny BPE model by hand and watch each merge.
3. Encode a new word with the merge list you learned.
4. Train the same idea with HuggingFace `tokenizers` (current API).
5. Inspect a production BPE vocabulary with `tiktoken`.


## Learning path

```mermaid
flowchart LR
  problem[OOV problem] --> scratch[From-scratch BPE]
  scratch --> encode[Encode with merges]
  encode --> hf[HuggingFace BpeTrainer]
  hf --> tik[tiktoken production BPE]
```

> **Diagram tip.** If your Jupyter UI shows a raw ` ```mermaid ` fence instead of a
> picture, that is a renderer gap (common in plain Classic Notebook). Read the
> flowchart as text, or open the notebook on GitHub / VS Code.
>
> Full-page schematics (Token Lab):
> [diagrams gallery](https://sourangshupal.github.io/tokenization-explainer/diagrams/)
> · local `../site/diagrams/index.html`

> Schematic: [BPE train loop](https://sourangshupal.github.io/tokenization-explainer/diagrams/03-bpe-train.html)
> · local [`../site/diagrams/03-bpe-train.html`](../site/diagrams/03-bpe-train.html)


## Why BPE exists

A **word-level** tokenizer stores every whole word. The moment a student types `unhappiness`
and that string never appeared in training, the model sees `[UNK]`. Spelling information is
gone.

BPE's bet: keep **frequent** words intact, and build rare words from **reusable pieces**.
`low` + `est` can form `lowest` even if `lowest` itself was rare.

The original NLP paper is Sennrich, Haddow, and Birch (2016). Neural nets had already used a
similar compression trick (Gage, 1994). GPT-2 later ran BPE on **UTF-8 bytes** so nothing is
unknown.


## The algorithm (picture)

```mermaid
flowchart TD
  corpus[Corpus of words plus counts] --> split[Split each word into characters]
  split --> endmark["Append an end-of-word mark </w>"]
  endmark --> count[Count every adjacent pair]
  count --> pick[Pick the pair with the highest count]
  pick --> merge[Glue that pair into one new symbol]
  merge --> vocab[Add the new symbol to the vocabulary]
  vocab --> enough{Reached N merges?}
  enough -->|no| count
  enough -->|yes| done[Save merge list and vocab]
```

`</w>` matters. Without it, BPE cannot tell the `st` inside `star` from the `st` at the end
of `widest`. The end mark is a boundary.

> **Diagram tip.** If your Jupyter UI shows a raw ` ```mermaid ` fence instead of a
> picture, that is a renderer gap (common in plain Classic Notebook). Read the
> flowchart as text, or open the notebook on GitHub / VS Code.
>
> Full-page schematics (Token Lab):
> [diagrams gallery](https://sourangshupal.github.io/tokenization-explainer/diagrams/)
> · local `../site/diagrams/index.html`


## Setup — paths and corpora


In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path

from IPython.display import display, Markdown
import ipywidgets as widgets

def find_root() -> Path:
    here = Path.cwd()
    for candidate in [here, here.parent]:
        if (candidate / "data" / "tiny_corpus.txt").exists():
            return candidate
    raise FileNotFoundError("Run the notebook from the repo root or the notebooks/ folder.")

ROOT = find_root()
CORPUS = ROOT / "data" / "tiny_corpus.txt"          # richer file for HuggingFace / SentencePiece
SENNRICH = ROOT / "data" / "sennrich_toy.txt"      # classic four-word set for from-scratch labs
ARTIFACTS = ROOT / "artifacts"
PRETRAINED = ROOT / "models" / "pretrained"
ARTIFACTS.mkdir(exist_ok=True)
print(f"course corpus: {CORPUS}")
print(f"Sennrich toy:  {SENNRICH}")
print()
print("Mermaid diagrams: GitHub and JupyterLab often render ```mermaid fences.")
print("If you see raw fences, read the flowchart as text — the algorithms still run.")


## Two corpora (do not mix them up)

| File | Role |
|---|---|
| `data/sennrich_toy.txt` | Classic four-word set. **From-scratch BPE below uses this.** Same default as the website lab. |
| `data/tiny_corpus.txt` | Richer English paragraphs. **HuggingFace / SentencePiece cells use this.** |

If you train on the course corpus and compare to the website Sennrich slider, merges will
differ. That is expected — not a bug.


In [ ]:
print("Sennrich toy:\n")
print(SENNRICH.read_text(encoding="utf-8"))
print("---")
print("Course corpus first 12 lines:\n")
print("\n".join(CORPUS.read_text(encoding="utf-8").splitlines()[:12]))


## From scratch: count words, then characters

We only keep alphabetic tokens so the first merges stay readable. Real BPE also sees
punctuation and digits. Production GPT BPE sees raw bytes.


In [ ]:
def load_word_counts(path: Path) -> Counter[str]:
    counts: Counter[str] = Counter()
    for line in path.read_text(encoding="utf-8").splitlines():
        for raw in line.lower().split():
            word = "".join(ch for ch in raw if ch.isalpha())
            if word:
                counts[word] += 1
    return counts


word_counts = load_word_counts(SENNRICH)
print(f"{len(word_counts)} word types, {sum(word_counts.values())} tokens")
print("Counts:", dict(word_counts))
print("Course-corpus types (HF later):", len(load_word_counts(CORPUS)))


In [ ]:
def initial_splits(counts: Counter[str]) -> dict[str, list[str]]:
    return {word: list(word) + ["</w>"] for word in counts}


splits = initial_splits(word_counts)
for word in ["low", "lower", "newest", "widest"]:
    if word in splits:
        print(f"{word:8}  {splits[word]}   x{word_counts[word]}")


## Count pairs and merge the winner

A **pair** is two neighbouring symbols inside a word. If `low` appears 5 times as
`['l','o','w','</w>']`, the pair `('l','o')` gets +5, not +1. Frequency is corpus count, not
type count.


In [ ]:
def pair_counts(splits: dict[str, list[str]], counts: Counter[str]) -> Counter[tuple[str, str]]:
    pairs: Counter[tuple[str, str]] = Counter()
    for word, freq in counts.items():
        symbols = splits[word]
        for left, right in zip(symbols, symbols[1:]):
            pairs[(left, right)] += freq
    return pairs


def apply_merge(splits: dict[str, list[str]], pair: tuple[str, str]) -> dict[str, list[str]]:
    a, b = pair
    glued = a + b
    updated: dict[str, list[str]] = {}
    for word, symbols in splits.items():
        out: list[str] = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                out.append(glued)
                i += 2
            else:
                out.append(symbols[i])
                i += 1
        updated[word] = out
    return updated


stats = pair_counts(splits, word_counts)
print("Top 10 pairs before any merge:")
for pair, freq in stats.most_common(10):
    print(f"  {pair!s:30} {freq}")


## Train: repeat the merge N times

Each round we record `(left, right) → left+right`. That ordered list **is** the tokenizer.
Encoding a new word later means replaying these merges in the same order.


In [ ]:
def train_bpe(
    counts: Counter[str],
    num_merges: int,
) -> tuple[list[tuple[str, str]], dict[str, list[str]]]:
    splits = initial_splits(counts)
    merges: list[tuple[str, str]] = []
    for step in range(1, num_merges + 1):
        stats = pair_counts(splits, counts)
        if not stats:
            break
        pair, freq = stats.most_common(1)[0]
        if freq < 2:
            print(f"stop at step {step}: best pair only appears {freq} time(s)")
            break
        merges.append(pair)
        splits = apply_merge(splits, pair)
        a, b = pair
        print(f"{step:02d}. merge {a!r} + {b!r}  →  {a + b!r}   (freq {freq})")
    return merges, splits


merges, trained_splits = train_bpe(word_counts, num_merges=25)
print(f"\nlearned {len(merges)} merges")
print("\nHow the classic words look after training:")
for word in ["low", "lower", "newest", "widest", "lowest"]:
    if word in trained_splits:
        print(f"  {word:8} {trained_splits[word]}")


## Encode a new word

Training never saw every possible word. Encoding still works: split into characters, then
apply **the same merges in the same order**. If `e` + `s` was merged during training, it
will merge here too.


In [ ]:
def encode_word(word: str, merges: list[tuple[str, str]]) -> list[str]:
    symbols = list(word.lower()) + ["</w>"]
    for pair in merges:
        symbols = apply_merge({"w": symbols}, pair)["w"]
    return symbols


def encode_text(text: str, merges: list[tuple[str, str]]) -> list[str]:
    pieces: list[str] = []
    for raw in text.split():
        word = "".join(ch for ch in raw.lower() if ch.isalpha())
        if word:
            pieces.extend(encode_word(word, merges))
        else:
            pieces.append(raw)
    return pieces


for sample in ["lowest", "newer", "tokenization", "unhappiness"]:
    print(f"{sample:15} → {encode_word(sample, merges)}")

# Golden check — Sennrich toy (~15 merges before pairs die out)
lowest_pieces = encode_word("lowest", merges)
print("\ngolden assert on lowest:", lowest_pieces)
assert lowest_pieces == ["low", "est</w>"], lowest_pieces
print("assert ok — lowest → low + est</w>")


## Faded example — fill merge #1

Below, merge 0 is annotated for you. Confirm merge 1 matches the golden pair (the cell
asserts). This is the scaffolding step before you re-run `train_bpe` with other budgets.


In [ ]:
# Faded example: you fill merge #1 after seeing merge #0 annotated.
from collections import Counter

def _pair_counts(splits: dict[str, list[str]], counts: Counter[str]) -> Counter[tuple[str, str]]:
    pairs: Counter[tuple[str, str]] = Counter()
    for word, freq in counts.items():
        symbols = splits[word]
        for left, right in zip(symbols, symbols[1:]):
            pairs[(left, right)] += freq
    return pairs

def _apply_merge(splits: dict[str, list[str]], pair: tuple[str, str]) -> dict[str, list[str]]:
    a, b = pair
    glued = a + b
    updated: dict[str, list[str]] = {}
    for word, symbols in splits.items():
        out: list[str] = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                out.append(glued)
                i += 2
            else:
                out.append(symbols[i])
                i += 1
        updated[word] = out
    return updated

toy = Counter({"low": 5, "lower": 2, "newest": 3, "widest": 2})
splits0 = {w: list(w) + ["</w>"] for w in toy}
pairs0 = _pair_counts(splits0, toy)
best0, freq0 = pairs0.most_common(1)[0]
print("Annotated merge 0 (already done for you):")
print(f"  best pair {best0!r} with freq {freq0}")
print("  → glue into", repr(best0[0] + best0[1]))
splits1 = _apply_merge(splits0, best0)

# YOUR TURN: pick the best pair after merge 0
pairs1 = _pair_counts(splits1, toy)
# student_pair = pairs1.most_common(1)[0][0]   # uncomment and run
student_pair = pairs1.most_common(1)[0][0]
print("Your merge 1 pair:", student_pair)

golden_pair = pairs1.most_common(1)[0][0]
assert student_pair == golden_pair, f"expected {golden_pair}, got {student_pair}"
print("assert ok — merge 1 matches the golden pair")


## Interactive playground

Type a phrase. The encoder uses **your** merge list from the cell above. Re-run training
with a different `num_merges` and this widget will still use whatever `merges` is in memory.


In [ ]:
box = widgets.Text(
    value="lowest newest unhappiness",
    description="Text:",
    layout=widgets.Layout(width="90%"),
    style={"description_width": "50px"},
)
out = widgets.Output()


def _run(_change=None) -> None:
    with out:
        out.clear_output()
        pieces = encode_text(box.value, merges)
        print("pieces:", pieces)
        print("count: ", len(pieces))


box.observe(_run, names="value")
_run()
display(box, out)


## Production library: HuggingFace `tokenizers`

The Rust library `tokenizers` (we installed **0.23.x** via uv) trains BPE at production
speed. The ideas are the same: a model, a pre-tokenizer, a trainer, then `encode`.

Current API — do not use old `BertTokenizer` constructors from `transformers` here. We stay
on the `tokenizers` package.


In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

hf_bpe = Tokenizer(BPE(unk_token="[UNK]"))
hf_bpe.pre_tokenizer = Whitespace()
trainer = BpeTrainer(
    special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]"],
    vocab_size=120,
    min_frequency=1,
    show_progress=False,
)
hf_bpe.train([str(CORPUS)], trainer)

encoded = hf_bpe.encode("lower newest unhappiness tokenization")
print("tokens:", encoded.tokens)
print("ids:   ", encoded.ids)
print("vocab size:", hf_bpe.get_vocab_size())


In [ ]:
hf_path = ARTIFACTS / "hf_bpe.json"
hf_bpe.save(str(hf_path))
reloaded = Tokenizer.from_file(str(hf_path))
print("reloaded:", reloaded.encode("widest lower").tokens)


## Production BPE: OpenAI `tiktoken`

You do not train `tiktoken`. OpenAI already ran byte-level BPE on a huge corpus and shipped
the merge table. `cl100k_base` is the GPT-4 / GPT-3.5 family. `o200k_base` is the newer
GPT-4o family. Notice how one English word is often **one** token, while a rare or
non-English string becomes several.

This is still BPE. The difference is scale and the byte-level base vocabulary.

**Campus / offline note.** The first `tiktoken.get_encoding(...)` call **downloads** the
encoding files. Do that once on a network (home wifi), then re-run offline. If the download
is blocked in class, skip this cell and use the HuggingFace BPE section above.


In [ ]:
import tiktoken

for name in ["cl100k_base", "o200k_base"]:
    enc = tiktoken.get_encoding(name)
    samples = [
        "tokenization",
        "unhappiness",
        "lowest newest",
        "বাংলা",
        "👋",
    ]
    print(f"\n=== {name}  (vocab ~ {enc.n_vocab}) ===")
    for s in samples:
        ids = enc.encode(s)
        pieces = [enc.decode([i]) for i in ids]
        print(f"  {s!r:20} ids={ids}  pieces={pieces!r}")


## Where you will see BPE

| System | Flavour |
|---|---|
| GPT-2 / GPT-3 / GPT-4 / GPT-4o | byte-level BPE (`tiktoken`) |
| Many Llama / Mistral tokenizers | BPE (SentencePiece or HuggingFace) |
| The mini lab on the website | character BPE with `</w>` |

BPE never asks “is this a linguistically nice morpheme?” It only asks “did this pair occur a
lot?” WordPiece (next notebook) changes that scoring rule.


## Exercises

1. Re-run `train_bpe` with `num_merges=5` and `num_merges=40`. Encode `lowest`. What changed?
2. Why does `</w>` exist? Try a thought experiment: merge `st` in both `star` and `widest`.
3. HuggingFace BPE used a `Whitespace` pre-tokenizer. What would break if you skipped it?
4. Using `tiktoken`, encode your name in English and in another script. Compare token counts.

Write answers in a new cell below. Instructor golden notes live in `INSTRUCTOR.md` at the
repo root — keep that file closed during the lab if you want an honest attempt.


## Capstone (after all three notebooks)

Given a model card, name the tokenizer family and one consequence for length:

| Card clue | Algorithm |
|---|---|
| BERT / `##` pieces / `[CLS]` | WordPiece |
| GPT-4o / `cl100k` / `o200k` / tiktoken | byte-level BPE |
| T5 / `spiece.model` / `▁` | SentencePiece (usually Unigram) |

Then encode the same prompt with `tiktoken` (`o200k_base`) and estimate how many
tokens a 4k-character English email costs vs the same email in Bengali.
